# z026 - Variables nuevas por grupos (`nuevas_variables_bajas_final.xlsx`)

**Referencia:** receta actual = trial 34 de z005, target juntar (BAJA+1 + BAJA+2), 761 variables (originales + lags/deltas 1 y 2).
Ganancia calculada afuera, solo con BAJA+2. 10 semillas × 2 cortes (igual que z024): train 202104 → valid 202106 ("junio")
y train 202103 → valid 202105 ("mayo").

| Versión | Qué agrega a la referencia |
|---|---|
| A | límite y uso de tarjetas (+ `sin_tarjeta`, de z025) + lags/deltas 1 y 2 de sus 4 montos m_ |
| B | plata del cliente + lags/deltas 1 y 2 de sus 11 montos m_ |
| C | uso y fidelidad + deltas 1 y 2 de c_trx_total, c_servicios_usados y c_trx_digitales |
| combo | los grupos que pasen la regla, juntos (solo si pasan 2 o más) |
| *_solo | la misma versión con target solo BAJA+2 (para confirmar que juntar sigue ganando) |

**Cambios respecto del Excel** (acordados con Keila): `sin_tarjeta` entra en A; `c_trx_digitales` = home banking + app
(la app vale 0/1, z025) y el call center queda aparte (ya es original) pero cuenta en `c_trx_total`; cocientes con
denominador ≤ 0 → NaN; la columna de débitos Visa es `mttarjeta_visa_debitos_automaticos` (doble t).

**Regla de adopción (definida antes de correr, la de z024):** gana en **≥ 8/10 semillas** (ganancia máxima, pareada por semilla)
**y** la ganancia media en **N fijo** (7,29% de los clientes del mes = 12.000/164.647) mejora, **en los dos cortes**.

**Ojo corte mayo:** entrena con 202103, primer mes de los datos → lags/deltas vacíos en el train y `meses_con_payroll_3` /
`meses_activo_seguidos` incompletos. Las variables de historia salen perjudicadas en ese corte.

**Chequeos extra:** importancia (gain) de cada versión y, por segmento (sin uso digital, sin sueldo, sin tarjeta),
qué % de las BAJA+2 queda dentro del N fijo: si un grupo gana en total pero pierde un segmento, le está tapando señal.

Cada predicción se guarda en `probs_z026/`: si se corta, volver a correr y retoma. Progreso en `z026_progreso.txt`.

In [1]:
# --- 1. Librerías y parámetros (igual que z024) ---
import os, time, shutil
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import wilcoxon

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
SEMILLAS = [270131, 417293, 583117, 692347, 851071,
            135791, 246809, 314159, 527183, 964201]
CORTES = {"junio": (202104, 202106), "mayo": (202103, 202105)}   # nombre: (mes train, mes valid)
PROP_N = 12_000 / 164_647
CARPETA, CARPETA_Z024 = "probs_z026", "probs_z024"
os.makedirs(CARPETA, exist_ok=True)

params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}
elegidos = pd.read_csv("z005_top10_final.csv", sep=";", decimal=",", encoding="utf-8-sig", index_col="trial")
trial = elegidos.loc[34]
params = {**params_fijos, "num_leaves": int(trial["params_num_leaves"]),
          "min_sum_hessian_in_leaf": float(trial["params_min_sum_hessian_in_leaf"])}
N_ARBOLES = int(trial["user_attrs_best_iter"])

def log(texto):
    with open("z026_progreso.txt", "a", encoding="utf-8") as f:
        f.write(time.strftime("%H:%M:%S ") + texto + "\n")
log(f"inicio | árboles {N_ARBOLES}")
params, N_ARBOLES

({'objective': 'binary',
  'learning_rate': 0.005,
  'min_data_in_leaf': 0,
  'feature_fraction': 0.5,
  'feature_fraction_bynode': 0.2,
  'max_bin': 31,
  'verbose': -1,
  'num_leaves': 137,
  'min_sum_hessian_in_leaf': 9.755574121446362},
 538)

In [2]:
# --- 2. Datos (202103-202106) ---
df = pd.read_parquet("../competencia_01.parquet")
df = df[df["foto_mes"] <= 202106]
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
df["y_juntar"] = df["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)
df["y_solo"]   = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "y_juntar", "y_solo"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]   # las originales (las únicas con lags de siempre)
print(len(cols), "originales")

152 originales


In [3]:
# --- 3. Variables nuevas (ANTES del paso de lags) ---
def s(*columnas):
    # Suma con fillna(0) en cada término: si no tiene Master, sus columnas son NaN y la suma daría NaN
    total = 0
    for c in columnas:
        total = total + (df[c].fillna(0) if isinstance(c, str) else c.fillna(0))
    return total

def div(a, b):
    # Cociente: si el denominador es 0 o negativo, NaN. Ej.: 1.000 dólares / saldo −5.000 → NaN (no tiene sentido)
    return (a / b.where(b > 0)).astype("float32")

n = {}
# ---------- Grupo A: límite y uso de tarjetas ----------
n["m_limite_total"]     = s("Visa_mlimitecompra", "Master_mlimitecompra")
n["m_saldo_tarjetas"]   = s("Visa_msaldototal", "Master_msaldototal")
n["m_tarjetas_consumo"] = s("Visa_mconsumototal", "Master_mconsumototal")
n["r_saldo_limite"]     = div(n["m_saldo_tarjetas"], n["m_limite_total"])
n["r_consumo_limite"]   = div(n["m_tarjetas_consumo"], n["m_limite_total"])
n["r_pago_saldo"]       = div(s("Visa_mpagado", "Master_mpagado"), n["m_saldo_tarjetas"])
n["c_tarjetas_trx"]     = s("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones")
n["m_adelantos_total"]  = s("Visa_madelantopesos", "Visa_madelantodolares", "Master_madelantopesos", "Master_madelantodolares")
n["status_tarjeta_max"] = df[["Visa_status", "Master_status"]].max(axis=1)
n["sin_tarjeta"]        = (n["m_limite_total"] <= 0).astype("int8")          # z025: 5,1× el promedio de BAJA+2

# ---------- Grupo B: plata del cliente ----------
n["m_saldo_paquete"]     = s("mcuenta_corriente", "mcaja_ahorro")
n["m_saldo_adicional"]   = s("mcuenta_corriente_adicional", "mcaja_ahorro_adicional")
n["r_saldo_dolares"]     = div(df["mcaja_ahorro_dolares"], df["mcuentas_saldo"])
n["m_inversiones_total"] = s("mplazo_fijo_pesos", "mplazo_fijo_dolares", "minversion1_pesos", "minversion1_dolares", "minversion2")
n["m_patrimonio_banco"]  = s("mcuentas_saldo", n["m_inversiones_total"])
n["m_forex_neto"]        = df["mforex_buy"].fillna(0) - df["mforex_sell"].fillna(0)
n["m_payroll_total"]     = s("mpayroll", "mpayroll2")
n["m_ingresos_total"]    = s(n["m_payroll_total"], "mtransferencias_recibidas", "mcheques_depositados")
n["m_egresos_total"]     = s("mtransferencias_emitidas", "mcheques_emitidos", "mextraccion_autoservicio",
                             "mpagodeservicios", "mpagomiscuentas", "mcuenta_debitos_automaticos")
n["m_flujo_neto"]        = n["m_ingresos_total"] - n["m_egresos_total"]
n["m_debitos_aut_total"] = s("mcuenta_debitos_automaticos", "mttarjeta_visa_debitos_automaticos", "mttarjeta_master_debitos_automaticos")
n["m_descuentos_total"]  = s("mcajeros_propios_descuentos", "mtarjeta_visa_descuentos", "mtarjeta_master_descuentos")
n["c_cheques_rechazados"] = s("ccheques_emitidos_rechazados", "ccheques_depositados_rechazados")

# ---------- Grupo C: uso y fidelidad ----------
n["c_inversiones_total"]  = s("cplazo_fijo", "cinversion1", "cinversion2")
n["c_seguros_total"]      = s("cseguro_vida", "cseguro_auto", "cseguro_vivienda", "cseguro_accidentes_personales")
n["c_payroll_total"]      = s("cpayroll_trx", "cpayroll2_trx")
n["tiene_payroll"]        = (n["c_payroll_total"] > 0).astype("int8")
n["c_debitos_aut_total"]  = s("ccuenta_debitos_automaticos", "ctarjeta_visa_debitos_automaticos", "ctarjeta_master_debitos_automaticos")
n["c_pagos_servicios_total"] = s("cpagodeservicios", "cpagomiscuentas", n["c_debitos_aut_total"])
n["c_descuentos_total"]   = s("ccajeros_propios_descuentos", "ctarjeta_visa_descuentos", "ctarjeta_master_descuentos")
n["c_comisiones_total"]   = s("ccomisiones_mantenimiento", "ccomisiones_otras")
n["c_trx_digitales"]      = s("chomebanking_transacciones", "cmobile_app_trx")     # la app vale 0/1: suma 1 si la usó
n["c_trx_presenciales"]   = s("ccajas_transacciones", "catm_trx", "catm_trx_other")
n["r_atm_otros"]          = div(df["catm_trx_other"].fillna(0), s("catm_trx", "catm_trx_other"))
n["c_trx_total"]          = s(n["c_trx_digitales"], n["c_trx_presenciales"], "ctarjeta_debito_transacciones",
                              n["c_tarjetas_trx"], "ccallcenter_transacciones")   # el call center cuenta como actividad
usados = [df[c] for c in ["ctarjeta_debito_transacciones", "cpagodeservicios", "cpagomiscuentas", "chomebanking_transacciones",
                          "cmobile_app_trx", "catm_trx", "ccajas_transacciones", "ctransferencias_emitidas",
                          "ctransferencias_recibidas", "cforex"]] + \
         [n[c] for c in ["c_tarjetas_trx", "c_payroll_total", "c_debitos_aut_total", "m_inversiones_total", "c_seguros_total"]]
n["c_servicios_usados"]   = sum((x > 0).astype("int8") for x in usados)            # 0 a 15 (NaN cuenta como no usado)
n["r_uso_vs_tenencia"]    = div(n["c_servicios_usados"].astype("float32"), df["cproductos"])
n["banco_principal"]      = ((n["tiene_payroll"] == 1) & (n["c_debitos_aut_total"] > 0) & (n["c_tarjetas_trx"] > 0)).astype("int8")

cliente = df["numero_de_cliente"]
tp = n["tiene_payroll"]
n["meses_con_payroll_3"] = (tp + tp.groupby(cliente).shift(1).fillna(0) + tp.groupby(cliente).shift(2).fillna(0)).astype("int8")
# Racha de meses seguidos activo: cada mes inactivo abre un "bloque" nuevo; dentro del bloque se suman los meses activos
activo = (n["c_servicios_usados"] > 0).astype("int16")
bloque = (activo == 0).groupby(cliente).cumsum()
n["meses_activo_seguidos"] = activo.groupby([cliente, bloque]).cumsum().astype("int8")

nuevas_df = pd.DataFrame(n).astype("float32")
del n, usados

GRUPO_A = ["m_limite_total", "m_saldo_tarjetas", "m_tarjetas_consumo", "r_saldo_limite", "r_consumo_limite", "r_pago_saldo",
           "c_tarjetas_trx", "m_adelantos_total", "status_tarjeta_max", "sin_tarjeta"]
GRUPO_B = ["m_saldo_paquete", "m_saldo_adicional", "r_saldo_dolares", "m_patrimonio_banco", "m_inversiones_total", "m_forex_neto",
           "m_payroll_total", "m_ingresos_total", "m_egresos_total", "m_flujo_neto", "m_debitos_aut_total", "m_descuentos_total",
           "c_cheques_rechazados"]
GRUPO_C = ["c_inversiones_total", "c_seguros_total", "c_payroll_total", "tiene_payroll", "c_debitos_aut_total",
           "c_pagos_servicios_total", "c_descuentos_total", "c_comisiones_total", "c_trx_digitales", "c_trx_presenciales",
           "r_atm_otros", "c_trx_total", "c_servicios_usados", "r_uso_vs_tenencia", "banco_principal",
           "meses_con_payroll_3", "meses_activo_seguidos"]
assert sorted(GRUPO_A + GRUPO_B + GRUPO_C) == sorted(nuevas_df.columns)
nuevas_df.describe().T[["count", "mean", "min", "max"]]

,count,mean,min,max
m_limite_total,654066.0,721926.500000,0.000000e+00,1.887085e+07
m_saldo_tarjetas,654066.0,46567.187500,-9.745695e+05,8.725745e+06
m_tarjetas_consumo,654066.0,24713.792969,-4.483418e+05,9.261745e+06
r_saldo_limite,627497.0,0.114000,-1.217342e+02,1.581497e+02
r_consumo_limite,627497.0,0.054463,-4.222809e-01,2.697693e+01
r_pago_saldo,565152.0,0.157459,0.000000e+00,4.174226e+02
c_tarjetas_trx,654066.0,15.184455,0.000000e+00,7.230000e+02
m_adelantos_total,654066.0,38.077393,0.000000e+00,1.983619e+05
status_tarjeta_max,627497.0,0.037237,0.000000e+00,9.000000e+00
sin_tarjeta,654066.0,0.040621,0.000000e+00,1.000000e+00


In [4]:
# --- 4. Lags y deltas: los de siempre (solo originales) + los de las nuevas, con la lista armada a mano ---
grupo = df.groupby("numero_de_cliente")[cols]
partes = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    partes += [lag, delta]
df = pd.concat([df] + partes, axis=1)
del partes, lag, delta, grupo
base = [c for c in df.columns if c not in no_usar]    # las 761 de siempre (mismo orden que z024)
print("Variables base:", len(base))

M_CON_LAGS = [c for c in GRUPO_A + GRUPO_B if c.startswith("m_")]       # 15 montos: lag 1, lag 2, delta 1, delta 2
C_CON_DELTAS = ["c_trx_total", "c_servicios_usados", "c_trx_digitales"]   # solo delta 1 y delta 2
g_nuevas = nuevas_df.groupby(df["numero_de_cliente"])
historia = {}
for k in [1, 2]:
    lag_m = g_nuevas[M_CON_LAGS].shift(k)
    for c in M_CON_LAGS:
        historia[f"{c}_lag{k}"] = lag_m[c]
        historia[f"{c}_delta{k}"] = nuevas_df[c] - lag_m[c]
    lag_c = g_nuevas[C_CON_DELTAS].shift(k)
    for c in C_CON_DELTAS:
        historia[f"{c}_delta{k}"] = nuevas_df[c] - lag_c[c]
historia = pd.DataFrame(historia).astype("float32")
df = pd.concat([df, nuevas_df, historia], axis=1)

def con_historia(lista):
    # la lista del grupo + sus lags/deltas (los que existan)
    return lista + [h for h in historia.columns if h.rsplit("_", 1)[0] in lista]
VARS_A, VARS_B, VARS_C = con_historia(GRUPO_A), con_historia(GRUPO_B), con_historia(GRUPO_C)
TODAS_NUEVAS = VARS_A + VARS_B + VARS_C
print(f"A: {len(VARS_A)}  B: {len(VARS_B)}  C: {len(VARS_C)}  (historia: {historia.shape[1]})")
del nuevas_df, historia, g_nuevas, lag_m, lag_c
log("variables listas")

Variables base: 761
A: 26  B: 57  C: 23  (historia: 66)


In [5]:
# --- 5. Versiones ---
VERSIONES = {                      # nombre: (variables, target)
    "ref":     (base, "y_juntar"),
    "A":       (base + VARS_A, "y_juntar"),
    "B":       (base + VARS_B, "y_juntar"),
    "C":       (base + VARS_C, "y_juntar"),
    "ref_solo": (base, "y_solo"),
    "A_solo":  (base + VARS_A, "y_solo"),
    "B_solo":  (base + VARS_B, "y_solo"),
    "C_solo":  (base + VARS_C, "y_solo"),
}
for v, (vs, t) in VERSIONES.items():
    print(f"{v:9s} {len(vs):5d} variables, target {t}")

foto = df["foto_mes"].values

def entrenar(version, semillas=SEMILLAS):
    # Igual que z024: un Dataset por versión y corte (los bines se arman con la primera semilla) y retoma lo guardado
    vs, target = VERSIONES[version]
    for corte, (mes_train, mes_valid) in CORTES.items():
        faltan = [s for s in semillas if not os.path.exists(f"{CARPETA}/{version}_{corte}_{s}.npy")]
        if not faltan:
            continue
        tr, va = foto == mes_train, foto == mes_valid
        datos = lgb.Dataset(df.loc[tr, vs], label=df.loc[tr, target])
        X_valid = df.loc[va, vs]
        for s in faltan:
            inicio = time.time()
            modelo = lgb.train({**params, "seed": s}, datos, num_boost_round=N_ARBOLES)
            np.save(f"{CARPETA}/{version}_{corte}_{s}.npy", modelo.predict(X_valid).astype("float32"))
            pd.Series(modelo.feature_importance("gain"), index=vs).to_frame("gain").to_parquet(f"{CARPETA}/imp_{version}_{corte}_{s}.parquet")
            log(f"{corte} {version} semilla {s}: {time.time() - inicio:.0f} s")
        del datos, X_valid

ref         761 variables, target y_juntar
A           787 variables, target y_juntar
B           818 variables, target y_juntar
C           784 variables, target y_juntar
ref_solo    761 variables, target y_solo
A_solo      787 variables, target y_solo
B_solo      818 variables, target y_solo
C_solo      784 variables, target y_solo


In [6]:
# --- 6. ref_solo: reusar z024 (c_solo_baja2) si al reentrenar la 1ª semilla da idéntico ---
s0 = SEMILLAS[0]
for corte in CORTES:
    if all(os.path.exists(f"{CARPETA}/ref_solo_{corte}_{s}.npy") for s in SEMILLAS):
        continue
    entrenar("ref_solo", semillas=[s0])                    # entrena solo la 1ª semilla de este y del otro corte si falta
    iguales = np.array_equal(np.load(f"{CARPETA}/ref_solo_{corte}_{s0}.npy"),
                             np.load(f"{CARPETA_Z024}/c_solo_baja2_{corte}_{s0}.npy"))
    log(f"verificación ref_solo {corte}: {'idéntico a z024' if iguales else 'DISTINTO a z024 → se reentrena'}")
    print(corte, "idéntico a z024:", iguales)
    if iguales:
        for s in SEMILLAS[1:]:
            shutil.copy(f"{CARPETA_Z024}/c_solo_baja2_{corte}_{s}.npy", f"{CARPETA}/ref_solo_{corte}_{s}.npy")
entrenar("ref_solo")      # no hace nada si ya se copiaron; si no dio idéntico, entrena lo que falta

junio idéntico a z024: True
mayo idéntico a z024: True


In [7]:
# --- 7. Entrenar ref (para tener su importancia), A, B, C y sus versiones solo BAJA+2 ---
for version in ["ref", "A", "B", "C", "A_solo", "B_solo", "C_solo"]:
    entrenar(version)
# Chequeo: ref reentrenada tiene que dar igual que la de z024
iguales = sum(np.array_equal(np.load(f"{CARPETA}/ref_{c}_{s}.npy"), np.load(f"{CARPETA_Z024}/ref_{c}_{s}.npy"))
              for c in CORTES for s in SEMILLAS)
log(f"entrenamiento terminado | ref idéntica a z024 en {iguales}/20")
print("ref idéntica a z024 en", iguales, "de 20 modelos")

ref idéntica a z024 en 20 de 20 modelos


In [8]:
# --- 8. Ganancias por semilla y comparaciones ---
def ganancias(versiones):
    filas = []
    for corte, (mes_train, mes_valid) in CORTES.items():
        va = foto == mes_valid
        gan = np.where(df.loc[va, "clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
        n_fijo = round(PROP_N * va.sum())
        for version in versiones:
            for s in SEMILLAS:
                prob = np.load(f"{CARPETA}/{version}_{corte}_{s}.npy")
                acum = np.cumsum(gan[np.argsort(-prob)])
                filas.append({"corte": corte, "version": version, "semilla": s, "n_fijo": n_fijo,
                              "gan_max": acum.max() / 1e6, "mejor_n": acum.argmax() + 1,
                              "gan_nfijo": acum[n_fijo - 1] / 1e6, "gan_12000": acum[11_999] / 1e6})
    return pd.DataFrame(filas)

def comparar(por_semilla, pares):
    # pares: {version: contra_quien}. Diferencias pareadas por semilla, en cada corte por separado
    idx = por_semilla.set_index(["version", "corte", "semilla"])
    res = []
    for version, contra in pares.items():
        for corte in CORTES:
            g, r = idx.loc[(version, corte)], idx.loc[(contra, corte)]
            d_max, d_fijo = g["gan_max"] - r["gan_max"], g["gan_nfijo"] - r["gan_nfijo"]
            res.append({"version": version, "contra": contra, "corte": corte,
                        "gan_max": g["gan_max"].mean(), "gan_max_desv": g["gan_max"].std(),
                        "gan_nfijo": g["gan_nfijo"].mean(), "gan_12000": g["gan_12000"].mean(), "mejor_n": g["mejor_n"].mean(),
                        "dif_max": d_max.mean(), "gana_max": (d_max > 0).sum(),
                        "p_max": np.nan if (d_max == 0).all() else wilcoxon(d_max).pvalue,
                        "dif_nfijo": d_fijo.mean(), "gana_nfijo": (d_fijo > 0).sum()})
    return pd.DataFrame(res).set_index(["version", "corte"])

def adopta(res, version):
    return all(res.loc[(version, c), "gana_max"] >= 8 and res.loc[(version, c), "dif_nfijo"] > 0 for c in CORTES)

GRUPOS = ["A", "B", "C"]
por_semilla = ganancias(list(VERSIONES))
res_grupos = comparar(por_semilla, {g: "ref" for g in GRUPOS})
res_grupos["adopta"] = [("SÍ" if adopta(res_grupos, v) else "no") for v, c in res_grupos.index]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
res_grupos.round(3)

C:\Users\keila\AppData\Local\Temp\ipykernel_33492\2181672366.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  g, r = idx.loc[(version, corte)], idx.loc[(contra, corte)]
C:\Users\keila\AppData\Local\Temp\ipykernel_33492\2181672366.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  g, r = idx.loc[(version, corte)], idx.loc[(contra, corte)]


contra  gan_max  gan_max_desv  gan_nfijo  gan_12000  mejor_n  dif_max  gana_max  p_max  dif_nfijo  gana_nfijo adopta
version corte                                                                                                                     
A       junio    ref  414.873         4.461    385.962     386.43  14325.7   -3.954         2  0.037      -1.54           5     no
        mayo     ref  277.197         1.974    253.220     252.45   7316.1    2.731         9  0.004       2.64           7     no
B       junio    ref  422.661         3.777    398.173     398.64  14214.5    3.834         8  0.131      10.67          10     no
        mayo     ref  273.699         2.795    249.260     248.49   7795.3   -0.767         4  0.695      -1.32           4     no
C       junio    ref  423.255         4.047    399.822     401.17  14704.9    4.427         8  0.064      12.32           9     no
        mayo     ref  273.545         1.747    246.620     246.18   7456.9   -0.921         5  0.375      -3.96           1     no

In [9]:
# --- 8b. Método del tutor: promediar las curvas de ganancia de las 10 semillas y tomar el máximo de la curva promedio ---
def curva_promedio(versiones):
    filas = []
    for corte, (mes_train, mes_valid) in CORTES.items():
        va = foto == mes_valid
        gan = np.where(df.loc[va, "clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
        for version in versiones:
            prom = np.mean([np.cumsum(gan[np.argsort(-np.load(f"{CARPETA}/{version}_{corte}_{s}.npy"))]) for s in SEMILLAS], axis=0)
            filas.append({"version": version, "corte": corte, "max_curva_prom": prom.max() / 1e6, "n_max": prom.argmax() + 1})
    return pd.DataFrame(filas).pivot(index="version", columns="corte").round(1)
curva_promedio(list(VERSIONES))

max_curva_prom         n_max      
corte             junio   mayo  junio  mayo
version                                    
A                 411.2  273.8  14358  7458
A_solo            395.9  251.3  14577  7135
B                 419.2  270.8  14427  8104
B_solo            401.0  248.0  14112  7718
C                 418.8  269.6  14719  6987
C_solo            401.0  244.9  13427  8223
ref               415.9  271.3  14989  6785
ref_solo          396.0  249.5  14107  7713

In [10]:
# --- 9. Si pasan 2 o más grupos: probarlos juntos (combo), también con solo BAJA+2 ---
ganadores = [g for g in GRUPOS if adopta(res_grupos, g)]
print("Grupos que pasan la regla:", ganadores or "ninguno")
if len(ganadores) >= 2:
    vars_combo = base + sum([{"A": VARS_A, "B": VARS_B, "C": VARS_C}[g] for g in ganadores], [])
    VERSIONES["combo"] = (vars_combo, "y_juntar")
    VERSIONES["combo_solo"] = (vars_combo, "y_solo")
    entrenar("combo"); entrenar("combo_solo")
    log("combo terminado")
    por_semilla = ganancias(list(VERSIONES))
    res_combo = comparar(por_semilla, {"combo": "ref"})
    res_combo["adopta"] = "SÍ" if adopta(res_combo, "combo") else "no"
    display(res_combo.round(3))
por_semilla.to_csv("z026_por_semilla.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")

Grupos que pasan la regla: ninguno


In [11]:
# --- 10. ¿Juntar sigue ganando con las variables nuevas? (juntar contra solo BAJA+2, misma versión, semilla a semilla) ---
JUNTAR = [v for v in VERSIONES if not v.endswith("_solo")]
res_solo = comparar(por_semilla, {v: f"{v}_solo" for v in JUNTAR})
res_solo.rename(columns={"gana_max": "juntar_gana_max", "gana_nfijo": "juntar_gana_nfijo"}).round(3)

C:\Users\keila\AppData\Local\Temp\ipykernel_33492\2181672366.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  g, r = idx.loc[(version, corte)], idx.loc[(contra, corte)]
C:\Users\keila\AppData\Local\Temp\ipykernel_33492\2181672366.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  g, r = idx.loc[(version, corte)], idx.loc[(contra, corte)]


contra  gan_max  gan_max_desv  gan_nfijo  gan_12000  mejor_n  dif_max  juntar_gana_max  p_max  dif_nfijo  juntar_gana_nfijo
version corte                                                                                                                               
ref     junio  ref_solo  418.828         3.443    387.502     387.75  14769.9   18.730               10  0.002      -4.07                  3
        mayo   ref_solo  274.466         1.980    250.580     250.14   8011.4   20.848               10  0.002      19.25                 10
A       junio    A_solo  414.873         4.461    385.962     386.43  14325.7   13.593                9  0.004      -6.49                  2
        mayo     A_solo  277.197         1.974    253.220     252.45   7316.1   22.941               10  0.002      23.21                 10
B       junio    B_solo  422.661         3.777    398.173     398.64  14214.5   18.590               10  0.002       4.40                  8
        mayo     B_solo  273.699         2.795    249.260     248.49   7795.3   21.717               10  0.002      11.22                 10
C       junio    C_solo  423.255         4.047    399.822     401.17  14704.9   17.339               10  0.002       1.65                  7
        mayo     C_solo  273.545         1.747    246.620     246.18   7456.9   24.945               10  0.002      17.82                 10

In [12]:
# --- 11. Segmentos: ¿un grupo tapa señal? % de las BAJA+2 de cada segmento que queda dentro del N fijo ---
seg_def = {
    "todos":       np.ones(len(df), bool),
    "sin_digital": (df["chomebanking_transacciones"].fillna(0) == 0) & (df["cmobile_app_trx"].fillna(0) == 0),
    "sin_sueldo":  df["c_payroll_total"] == 0,
    "sin_tarjeta": df["sin_tarjeta"] == 1,
}
filas = []
for corte, (mes_train, mes_valid) in CORTES.items():
    va = foto == mes_valid
    es_b2 = df.loc[va, "clase_ternaria"].values == "BAJA+2"
    n_fijo = round(PROP_N * va.sum())
    for version in JUNTAR:
        for s in SEMILLAS:
            prob = np.load(f"{CARPETA}/{version}_{corte}_{s}.npy")
            elegido = np.zeros(va.sum(), bool); elegido[np.argsort(-prob)[:n_fijo]] = True
            for seg, m in seg_def.items():
                m = np.asarray(m)[va]
                filas.append({"corte": corte, "version": version, "segmento": seg, "semilla": s,
                              "b2_segmento": (es_b2 & m).sum(), "captura": (elegido & es_b2 & m).sum() / (es_b2 & m).sum()})
seg = pd.DataFrame(filas)
captura = (seg.groupby(["corte", "segmento", "version"])["captura"].mean() * 100).unstack("version")
for v in JUNTAR:
    if v != "ref":
        captura[f"dif_{v}"] = captura[v] - captura["ref"]
print("BAJA+2 por segmento:", seg.groupby(["corte", "segmento"])["b2_segmento"].first().to_dict())
captura.round(1)        # % de BAJA+2 del segmento dentro del N fijo, promedio de 10 semillas

BAJA+2 por segmento: {('junio', 'sin_digital'): 356, ('junio', 'sin_sueldo'): 1023, ('junio', 'sin_tarjeta'): 285, ('junio', 'todos'): 1098, ('mayo', 'sin_digital'): 234, ('mayo', 'sin_sueldo'): 802, ('mayo', 'sin_tarjeta'): 132, ('mayo', 'todos'): 870}


version               A     B     C   ref  dif_A  dif_B  dif_C
corte segmento                                                
junio sin_digital  72.9  75.4  74.9  73.8   -1.0    1.5    1.0
      sin_sueldo   63.0  64.0  64.3  63.1   -0.1    0.9    1.1
      sin_tarjeta  61.4  64.2  63.3  62.7   -1.3    1.5    0.6
      todos        59.2  60.2  60.3  59.3   -0.1    0.9    1.0
mayo  sin_digital  83.8  84.4  83.6  84.0   -0.2    0.4   -0.4
      sin_sueldo   65.0  64.5  64.5  64.8    0.3   -0.3   -0.2
      sin_tarjeta  86.1  85.3  84.7  85.2    0.9    0.2   -0.5
      todos        60.8  60.3  60.1  60.5    0.3   -0.1   -0.4

In [13]:
# --- 12. Importancia (gain promedio de las 10 semillas, cada corte por separado) ---
def importancia(version, corte):
    imp = pd.concat([pd.read_parquet(f"{CARPETA}/imp_{version}_{corte}_{s}.parquet")["gain"] for s in SEMILLAS], axis=1).mean(axis=1)
    return (imp / imp.sum() * 100).sort_values(ascending=False)          # % del gain total

ORIG_CLAVE = ["ctrx_quarter", "mpasivos_margen", "cpayroll_trx", "mcuentas_saldo"]
resumen_imp, robo = [], []
for corte in CORTES:
    imp_ref = importancia("ref", corte)
    for v in [x for x in JUNTAR if x != "ref"]:
        imp = importancia(v, corte)
        nuevas_v = [c for c in imp.index if c in TODAS_NUEVAS]
        puesto = pd.Series(range(1, len(imp) + 1), index=imp.index)
        resumen_imp.append({"version": v, "corte": corte, "pct_gain_nuevas": imp[nuevas_v].sum(),
                            "nuevas": len(nuevas_v), "nuevas_gain_0": (imp[nuevas_v] == 0).sum()})
        for c in ORIG_CLAVE:
            robo.append({"version": v, "corte": corte, "variable": c, "pct_ref": imp_ref[c], "pct_version": imp[c],
                         "cambio_rel_%": (imp[c] / imp_ref[c] - 1) * 100})
        if corte == "junio":
            print(f"\n=== {v} (junio): top 15 de las nuevas, con su puesto en el ranking general de {len(imp)} ===")
            print(pd.DataFrame({"pct_gain": imp[nuevas_v].head(15).round(3), "puesto": puesto[nuevas_v].head(15)}).to_string())
            print("Gain 0 en junio:", list(imp[nuevas_v][imp[nuevas_v] == 0].index))
resumen_imp = pd.DataFrame(resumen_imp)
robo = pd.DataFrame(robo)
display(resumen_imp.round(2))
robo.round(2)


=== A (junio): top 15 de las nuevas, con su puesto en el ranking general de 787 ===
                           pct_gain  puesto
c_tarjetas_trx                2.310       8
m_saldo_tarjetas              1.334      17
status_tarjeta_max            1.223      19
r_saldo_limite                0.663      34
m_saldo_tarjetas_lag1         0.500      47
r_pago_saldo                  0.211     107
r_consumo_limite              0.171     123
m_limite_total_lag1           0.165     127
m_limite_total                0.158     132
m_tarjetas_consumo            0.122     148
m_tarjetas_consumo_lag1       0.105     154
m_saldo_tarjetas_delta1       0.091     162
m_limite_total_delta1         0.058     191
m_tarjetas_consumo_delta1     0.054     195
sin_tarjeta                   0.011     278
Gain 0 en junio: ['m_tarjetas_consumo_delta2', 'm_tarjetas_consumo_lag2', 'm_saldo_tarjetas_delta2', 'm_saldo_tarjetas_lag2', 'm_limite_total_delta2', 'm_limite_total_lag2', 'm_adelantos_total', 'm_adelantos_tot

,version,corte,pct_gain_nuevas,nuevas,nuevas_gain_0
0,A,junio,7.18,26,11
1,B,junio,20.21,57,24
2,C,junio,14.06,23,3
3,A,mayo,8.55,26,17
4,B,mayo,20.04,57,45
5,C,mayo,18.25,23,6


,version,corte,variable,pct_ref,pct_version,cambio_rel_%
0,A,junio,ctrx_quarter,5.63,5.62,-0.04
1,A,junio,mpasivos_margen,2.91,2.92,0.26
2,A,junio,cpayroll_trx,3.07,2.61,-15.03
3,A,junio,mcuentas_saldo,2.62,2.71,3.44
4,B,junio,ctrx_quarter,5.63,5.70,1.24
5,B,junio,mpasivos_margen,2.91,2.12,-27.07
6,B,junio,cpayroll_trx,3.07,1.67,-45.67
7,B,junio,mcuentas_saldo,2.62,1.54,-41.17
8,C,junio,ctrx_quarter,5.63,5.58,-0.88
9,C,junio,mpasivos_margen,2.91,2.88,-1.15


In [14]:
# --- 13. Anotar en resultados_experimentos.csv (una fila por versión y corte) ---
ruta = "resultados_experimentos.csv"
hist = pd.read_csv(ruta, sep=";", decimal=",", encoding="utf-8-sig")
hist = hist[~hist["descripcion"].str.startswith("z026")]      # sin duplicar si se vuelve a correr
todas = comparar(por_semilla, {v: ("ref" if not v.endswith("_solo") else v[:-5]) for v in VERSIONES})
nuevas_filas = []
for (version, corte), r in todas.iterrows():
    mes_train, mes_valid = CORTES[corte]
    if version == "ref":
        extra = ""
    elif version.endswith("_solo"):
        extra = f"; vs {r['contra']} (juntar): dif máx {r['dif_max']:+.1f} M, solo gana {int(r['gana_max'])}/10"
    else:
        extra = (f"; dif máx {r['dif_max']:+.1f} M, gana {int(r['gana_max'])}/10, p={r['p_max']:.3f}; "
                 f"dif N fijo {r['dif_nfijo']:+.1f} M, gana {int(r['gana_nfijo'])}/10; adopta: {'SÍ' if adopta(todas, version) else 'no'}")
    nuevas_filas.append({
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"z026 {version} ({corte}): trial 34, 10 semillas, {len(VERSIONES[version][0])} variables; N fijo {r['gan_nfijo']:.1f} M" + extra,
        "meses_train": str([mes_train]), "mes_valid": mes_valid, "semilla": str(SEMILLAS), "params": str(params),
        "mejor_n": int(r["mejor_n"]), "ganancia_max_M": round(r["gan_max"], 1), "ganancia_top12000_M": round(r["gan_12000"], 1)})
hist = pd.concat([hist, pd.DataFrame(nuevas_filas)], ignore_index=True)
hist["n_enviados"] = hist["n_enviados"].astype("Int64")
hist.to_csv(ruta, sep=";", decimal=",", index=False, encoding="utf-8-sig")
log("todo listo")
print("Anotado en", ruta)

Anotado en resultados_experimentos.csv


C:\Users\keila\AppData\Local\Temp\ipykernel_33492\2181672366.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  g, r = idx.loc[(version, corte)], idx.loc[(contra, corte)]
C:\Users\keila\AppData\Local\Temp\ipykernel_33492\2181672366.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  g, r = idx.loc[(version, corte)], idx.loc[(contra, corte)]


## Conclusiones (11/10, 15:00) — ningún grupo pasa la regla

Diferencia contra la referencia (juntar, 761 variables), 10 semillas. Ganancia máx / N fijo (M), semillas ganadas (máx / N fijo):

| Grupo | Junio (04→06) | Mayo (03→05) | Curva promedio jun / may (ref 415,9 / 271,3) | Adopta |
|---|---|---|---|---|
| A tarjetas | −4,0 / −1,5, 2/10 y 5/10 | +2,7 / +2,6, 9/10 y 7/10 (p=0,004) | 411,2 / 273,8 | **no** (pierde junio) |
| B plata | +3,8 / **+10,7**, 8/10 y 10/10 | −0,8 / −1,3, 4/10 y 4/10 | 419,2 / 270,8 | **no** (empata mayo) |
| C uso y fidelidad | +4,4 / **+12,3**, 8/10 y 9/10 | −0,9 / −4,0, 5/10 y 1/10 | 418,8 / 269,6 | **no** (empata/pierde mayo) |

- **B y C** ganan claro en junio, sobre todo en N fijo, y empatan o pierden un poco en mayo. Mayo entrena con 202103 (sin historia):
  ahí los lags de B y `meses_con_payroll_3` / `meses_activo_seguidos` de C no funcionan → parte de la diferencia puede ser artefacto.
  Con la regla definida antes, no se adoptan. Son candidatos para una prueba con otro corte que tenga historia en el train.
- **A** es al revés: gana en mayo (9/10) y pierde en junio (2/10). No es consistente.
- **Juntar vs solo BAJA+2:** juntar gana la ganancia máx en 9-10/10 semillas en todas las versiones y los dos cortes (p=0,002-0,004).
  Cuarta confirmación. (En N fijo de junio, solo BAJA+2 a veces queda mejor: su N óptimo es más chico.)
- **Segmentos:** ningún grupo tapa a otro. La captura de BAJA+2 sin uso digital, sin sueldo y sin tarjeta cambia ≤ 1,5 puntos (B y C la suben un poco en junio).
- **Importancia (junio):** las nuevas se llevan A 7%, B 20%, C 14% del gain. Fuertes: m_ingresos_total (puesto 3), c_servicios_usados (3),
  m_egresos_total (4), r_uso_vs_tenencia (7), c_tarjetas_trx (8). B y C le sacan gain a sus "parientes" originales
  (cpayroll_trx −45%, mcuentas_saldo −41% con B): es redistribución entre variables casi iguales, no pérdida de ganancia.
- **No aportan nada (gain 0):** m_adelantos_total y sus lags, c_cheques_rechazados, y **todos los lag2/delta2 nuevos**
  (con 1 mes de train, el mes −2 no existe en ningún corte). sin_tarjeta casi 0 (0,01%, puesto 278): el árbol ya la encuentra con las originales.
